# Tessera-PQC Tutorial 2: Side-Channel Analysis

This notebook covers side-channel analysis techniques:
- **Power Analysis**: CPA, DPA
- **Leakage Assessment**: TVLA (Test Vector Leakage Assessment)
- **Advanced Attacks**: Template attacks
- **Metrics**: SNR, Guessing Entropy

In [ ]:
import tessera as t
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Power Trace Generation

Side-channel attacks exploit information leaked during cryptographic operations.
The most common leakage model is the **Hamming Weight** model.

In [ ]:
# Generate simulated power traces
data = t.generate_traces(
    n_traces=1000,       # Number of traces to collect
    trace_length=32,     # Sample points per trace
    key_bytes=16,        # AES-128 style key
    noise_level=1.0,     # Gaussian noise standard deviation
    leakage_model="hw",  # Hamming Weight model
    seed=42
)

traces = data["traces"]
plaintexts = data["plaintexts"]
true_key = data["key"]

print(f"Traces shape:     {traces.shape}")
print(f"Plaintexts shape: {plaintexts.shape}")
print(f"True key:         {true_key.tolist()}")

In [ ]:
# Visualize some traces
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Plot individual traces
for i in range(5):
    axes[0].plot(traces[i], alpha=0.5, label=f"Trace {i}")
axes[0].set_xlabel("Sample")
axes[0].set_ylabel("Power")
axes[0].set_title("Individual Power Traces")
axes[0].legend()

# Plot mean trace
axes[1].plot(np.mean(traces, axis=0), 'b-', linewidth=2)
axes[1].fill_between(
    range(traces.shape[1]),
    np.mean(traces, axis=0) - np.std(traces, axis=0),
    np.mean(traces, axis=0) + np.std(traces, axis=0),
    alpha=0.3
)
axes[1].set_xlabel("Sample")
axes[1].set_ylabel("Power")
axes[1].set_title("Mean Trace with Standard Deviation")

plt.tight_layout()
plt.show()

## 2. Correlation Power Analysis (CPA)

CPA correlates the measured power traces with hypothetical power consumption
for each possible key value. The correct key produces the highest correlation.

In [ ]:
# Run CPA attack on first 4 key bytes
result = t.run_cpa(
    traces, 
    plaintexts, 
    key_bytes=[0, 1, 2, 3]  # Attack first 4 bytes
)

print("CPA Results:")
print(f"Recovered key: {result['recovered_key'].tolist()}")
print(f"True key:      {true_key[:4].tolist()}")
print(f"Confidences:   {result['confidences']}")

In [ ]:
# Visualize correlation matrix for first key byte
from tessera.attacks import CPA

attack = CPA()
best_key, confidence, _ = attack.attack_byte(traces, plaintexts, byte_index=0)
corr_matrix = attack.get_correlation_matrix()

plt.figure(figsize=(12, 4))

# Plot correlations for all key guesses
plt.subplot(121)
for kg in range(256):
    color = 'red' if kg == true_key[0] else 'gray'
    alpha = 1.0 if kg == true_key[0] else 0.1
    plt.plot(corr_matrix[kg], color=color, alpha=alpha)
plt.xlabel("Sample")
plt.ylabel("Correlation")
plt.title("CPA Correlations (red = correct key)")

# Plot max correlation per key guess
plt.subplot(122)
max_corr = np.max(np.abs(corr_matrix), axis=1)
plt.bar(range(256), max_corr, color='blue', alpha=0.5)
plt.axvline(true_key[0], color='red', linestyle='--', label=f"True key: {true_key[0]}")
plt.xlabel("Key Guess")
plt.ylabel("Max |Correlation|")
plt.title("Max Correlation per Key Guess")
plt.legend()

plt.tight_layout()
plt.show()

## 3. Differential Power Analysis (DPA)

DPA uses the difference of means between traces grouped by a bit of the intermediate value.

In [ ]:
# Run DPA attack
dpa_result = t.run_dpa(
    traces,
    plaintexts,
    key_bytes=[0, 1]
)

print("DPA Results:")
print(f"Recovered key: {dpa_result['recovered_key'].tolist()}")
print(f"True key:      {true_key[:2].tolist()}")

## 4. Test Vector Leakage Assessment (TVLA)

TVLA uses Welch's t-test to detect if an implementation leaks information.
A |t-value| > 4.5 indicates statistically significant leakage.

In [ ]:
# Generate traces for TVLA
# Group 1: Fixed plaintext (constant leakage)
# Group 2: Random plaintext (varying leakage)

from tessera.leakage import HammingWeightModel

model = HammingWeightModel(8)
n_traces = 500
trace_len = 32

# Fixed input group
fixed_key = np.array([0x12] * 8, dtype=np.uint8)
fixed_pt = np.array([0x34] * 8, dtype=np.uint8)
fixed_intermediate = fixed_pt ^ fixed_key

fixed_traces = []
for _ in range(n_traces):
    leakage = np.array([model(int(v)) for v in fixed_intermediate], dtype=np.float64)
    trace = np.zeros(trace_len)
    trace[:8] = leakage
    trace += np.random.randn(trace_len) * 0.5
    fixed_traces.append(trace)
fixed_traces = np.array(fixed_traces)

# Random input group
random_traces = []
for _ in range(n_traces):
    random_pt = np.random.randint(0, 256, 8, dtype=np.uint8)
    intermediate = random_pt ^ fixed_key
    leakage = np.array([model(int(v)) for v in intermediate], dtype=np.float64)
    trace = np.zeros(trace_len)
    trace[:8] = leakage
    trace += np.random.randn(trace_len) * 0.5
    random_traces.append(trace)
random_traces = np.array(random_traces)

print(f"Fixed traces shape:  {fixed_traces.shape}")
print(f"Random traces shape: {random_traces.shape}")

In [ ]:
# Run TVLA
tvla_result = t.run_tvla(fixed_traces, random_traces)

print("TVLA Results:")
print(f"Leakage detected: {tvla_result['leakage_detected']}")
print(f"Max |t-value|:    {tvla_result['max_t_value']:.2f}")
print(f"Threshold:        {tvla_result['threshold']:.2f}")

In [ ]:
# Visualize TVLA results
t_values = tvla_result['t_values']
threshold = tvla_result['threshold']

plt.figure(figsize=(12, 4))
plt.plot(t_values, 'b-', linewidth=1)
plt.axhline(threshold, color='r', linestyle='--', label=f'Threshold (+{threshold:.2f})')
plt.axhline(-threshold, color='r', linestyle='--', label=f'Threshold (-{threshold:.2f})')
plt.fill_between(range(len(t_values)), -threshold, threshold, alpha=0.1, color='green')
plt.xlabel("Sample")
plt.ylabel("t-value")
plt.title("TVLA Results (samples outside green zone indicate leakage)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Signal-to-Noise Ratio (SNR)

SNR measures the exploitable signal in the traces relative to noise.
Higher SNR = easier to attack.

In [ ]:
# Compute SNR using intermediate values as labels
labels = plaintexts[:, 0] ^ true_key[0]  # First byte intermediate values

snr = t.compute_snr(traces, labels)

plt.figure(figsize=(10, 4))
plt.plot(snr, 'b-', linewidth=1.5)
plt.xlabel("Sample")
plt.ylabel("SNR")
plt.title("Signal-to-Noise Ratio per Sample Point")
plt.grid(True, alpha=0.3)

# Highlight points of interest
poi = np.where(snr > np.max(snr) * 0.5)[0]
plt.scatter(poi, snr[poi], color='red', s=50, zorder=5, label=f'POI ({len(poi)} points)')
plt.legend()
plt.show()

print(f"Max SNR:  {np.max(snr):.4f}")
print(f"Mean SNR: {np.mean(snr):.4f}")

## 6. Attack Convergence Analysis

Analyze how many traces are needed for a successful attack.

In [ ]:
# Attack with increasing number of traces
from tessera.attacks import CPA

trace_counts = [50, 100, 200, 300, 500, 700, 1000]
success_rates = []
confidences = []

for n in trace_counts:
    attack = CPA()
    correct = 0
    conf_sum = 0
    
    for byte_idx in range(4):
        best_key, confidence, _ = attack.attack_byte(
            traces[:n], plaintexts[:n], byte_idx
        )
        if best_key == true_key[byte_idx]:
            correct += 1
        conf_sum += confidence
    
    success_rates.append(correct / 4)
    confidences.append(conf_sum / 4)
    print(f"n={n:4d}: success_rate={correct}/4, avg_confidence={conf_sum/4:.4f}")

In [ ]:
# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(trace_counts, success_rates, 'bo-', linewidth=2, markersize=8)
axes[0].axhline(1.0, color='g', linestyle='--', alpha=0.5)
axes[0].set_xlabel("Number of Traces")
axes[0].set_ylabel("Success Rate")
axes[0].set_title("Attack Success Rate vs. Number of Traces")
axes[0].grid(True, alpha=0.3)

axes[1].plot(trace_counts, confidences, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel("Number of Traces")
axes[1].set_ylabel("Average Confidence")
axes[1].set_title("Attack Confidence vs. Number of Traces")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Effect of Noise on Attack Success

Higher noise levels require more traces for successful attacks.

In [ ]:
# Test different noise levels
noise_levels = [0.1, 0.5, 1.0, 2.0, 3.0]
n_traces_test = 500

results_by_noise = []

for noise in noise_levels:
    data = t.generate_traces(
        n_traces=n_traces_test,
        trace_length=32,
        key_bytes=4,
        noise_level=noise,
        seed=42
    )
    
    result = t.run_cpa(data["traces"], data["plaintexts"], key_bytes=[0, 1, 2, 3])
    correct = np.sum(result["recovered_key"] == data["key"])
    
    results_by_noise.append({
        'noise': noise,
        'success': correct,
        'confidence': np.mean(result['confidences'])
    })
    print(f"Noise={noise:.1f}: {correct}/4 bytes correct, confidence={np.mean(result['confidences']):.4f}")

In [ ]:
# Plot noise effect
plt.figure(figsize=(10, 4))

plt.subplot(121)
plt.bar([r['noise'] for r in results_by_noise], 
        [r['success']/4 for r in results_by_noise],
        width=0.3, color='steelblue')
plt.xlabel("Noise Level")
plt.ylabel("Success Rate")
plt.title("Attack Success vs. Noise Level")
plt.ylim(0, 1.1)

plt.subplot(122)
plt.bar([r['noise'] for r in results_by_noise],
        [r['confidence'] for r in results_by_noise],
        width=0.3, color='coral')
plt.xlabel("Noise Level")
plt.ylabel("Average Confidence")
plt.title("Attack Confidence vs. Noise Level")

plt.tight_layout()
plt.show()

## Summary

In this notebook, you learned:

1. **Power Trace Generation**: Creating simulated traces with Hamming Weight leakage
2. **CPA Attack**: Correlating traces with hypothetical power to recover keys
3. **DPA Attack**: Using differential means to detect key-dependent leakage
4. **TVLA**: Detecting if an implementation leaks information
5. **SNR Analysis**: Measuring the exploitable signal in traces
6. **Attack Convergence**: How success improves with more traces
7. **Noise Effects**: How noise impacts attack success

**Key Takeaways**:
- More traces = better attack success
- Higher noise = need more traces
- SNR and TVLA help identify vulnerable implementations

**Next**: See Tutorial 3 for countermeasures against these attacks.